# Chapter 2.5: Data coverage and variant representation across samples
## Create AAV6-ML figures for Chapter 3 for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 22/04/2026
Finished: ...

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2
from scipy.stats import pearsonr, spearmanr, linregress
from matplotlib.patches import Patch


## 2. Preperation 
### 2.1. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for figures
figures_dir = general_dir/'figures'/'2.5_n_sample'
os.makedirs(figures_dir, exist_ok=True)

### 2.2. Define helper functions


In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)

In [ ]:
# sort file names from df_long
def sort_file_names (name_list):
    FILE_NAMES = {
        "gDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        },
        "cDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        }
    }
    
    for name in name_list:
        parts = name.split("_")
        n_parts = len(parts)
    
        # ext type
        if name.startswith("gDNA"):
            dna = "gDNA"
        elif name.startswith("cDNA"):
            dna = "cDNA"
        else:
            continue
    
        # Tissue 
        tissue = parts[1]
    
        if tissue not in ["heart", "liver"]:
            continue  
    
        # bio or tech
        if n_parts == 3:
            category = "biological"
        elif n_parts == 4:
            category = "technical"
        else:
            continue
    
        FILE_NAMES[dna][tissue][category].append(name)

    return FILE_NAMES 

In [ ]:
# comare and corr two tables
def compare_two_tables(
    df1,
    df2,
    name1="sample_1",
    name2="sample_2",
    compare_cols=None,
    key_col="AA_sequence",
    log_scale_cols=None,
    add_diagonal=True,
    alpha=0.25,
    point_size=8,
    figsize_per_panel=(5.5, 5),
    save=False,
    output_path=None,
    file_name=None,
    show=True
):
    """
    Compare two tables across one or multiple numeric columns by merging on key_col.

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Tables to compare
    name1, name2 : str
        Names shown in axis labels
    compare_cols : list of str
        Numeric columns to compare, e.g. ["Log2_enrichment", "RPM"]
    key_col : str
        Merge key, default = "AA_sequence"
    log_scale_cols : list of str
        Columns that should be plotted on log-log axes, e.g. ["RPM"]
    add_diagonal : bool
        Whether to add y=x diagonal
    alpha : float
        Point transparency
    point_size : int
        Scatter point size
    figsize_per_panel : tuple
        Width, height per subplot
    save : bool
        Whether to save figure
    output_path : str or Path
        Save directory
    file_name : str
        Output filename
    show : bool
        Whether to show plot

    Returns
    -------
    merged : pd.DataFrame
        Merged comparison table
    corr_df : pd.DataFrame
        Correlation summary table
    """

    if compare_cols is None:
        compare_cols = ["Log2_enrichment", "RPM"]

    if log_scale_cols is None:
        log_scale_cols = []

    # keep only needed columns
    cols1 = [key_col] + compare_cols
    cols2 = [key_col] + compare_cols

    a = df1[cols1].copy()
    b = df2[cols2].copy()

    # rename value columns
    a = a.rename(columns={col: f"{col}_{name1}" for col in compare_cols})
    b = b.rename(columns={col: f"{col}_{name2}" for col in compare_cols})

    # merge
    merged = a.merge(b, on=key_col, how="inner")

    # correlation results
    results = []

    # figure layout
    n = len(compare_cols)
    ncols = min(2, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows)
    )

    # make axes iterable
    if n == 1:
        axes = [axes]
    else:
        axes = np.array(axes).reshape(-1)

    for i, col in enumerate(compare_cols):
        ax = axes[i]

        xcol = f"{col}_{name1}"
        ycol = f"{col}_{name2}"

        sub = merged[[xcol, ycol]].dropna().copy()

        # correlations
        if len(sub) > 1:
            pearson_r, pearson_p = pearsonr(sub[xcol], sub[ycol])
            spearman_r, spearman_p = spearmanr(sub[xcol], sub[ycol])
        else:
            pearson_r, pearson_p = np.nan, np.nan
            spearman_r, spearman_p = np.nan, np.nan

        results.append({
            "column": col,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "n_points": len(sub)
        })

        # scatter
        sns.scatterplot(
            data=sub,
            x=xcol,
            y=ycol,
            s=point_size,
            alpha=alpha,
            ax=ax
        )

        # log scaling if requested
        if col in log_scale_cols:
            sub_pos = sub[(sub[xcol] > 0) & (sub[ycol] > 0)].copy()
            ax.clear()
            sns.scatterplot(
                data=sub_pos,
                x=xcol,
                y=ycol,
                s=point_size,
                alpha=alpha,
                ax=ax
            )
            ax.set_xscale("log")
            ax.set_yscale("log")
            plot_data = sub_pos
        else:
            plot_data = sub

        # diagonal
        if add_diagonal and len(plot_data) > 0:
            lims = [
                min(plot_data[xcol].min(), plot_data[ycol].min()),
                max(plot_data[xcol].max(), plot_data[ycol].max())
            ]
            ax.plot(lims, lims, "--", color="black", linewidth=1.2, alpha=0.6)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

        ax.set_xlabel(f"{col} ({name1})")
        ax.set_ylabel(f"{col} ({name2})")
        ax.set_title(col)

        ax.text(
            0.05, 0.95,
            f"Pearson r = {pearson_r:.4f}\nSpearman ρ = {spearman_r:.4f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

        sns.despine(ax=ax)

    # remove unused axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        import os
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            col_string = "_".join(compare_cols)
            file_name = f"compare_{name1}_vs_{name2}_{col_string}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    corr_df = pd.DataFrame(results)

    return merged, corr_df

## 3. Script Functions

### 3.1. variant presence distribution across variant presence

In [ ]:
def plot_variant_presence_distribution(
    table,
    tissue,
    extraction,
    col="n_samples_present",
    include_zero=False,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
    """
    Plot distribution of variants across the number of biological samples in which they are present.

    Parameters
    ----------
    table : pd.DataFrame
        Input table, e.g. df_pooled
    tissue : str
        Tissue name, e.g. 'liver'
    extraction : str
        Extraction type, e.g. 'gDNA' or 'cDNA'
    include_zero : bool
        Whether to include variants present in 0 samples
    title : bool
        Whether to show title
    save : bool
        Whether to save figure
    output_path : str or Path
        Output folder
    file_name : str
        Output filename
    """

    col="n_samples_present"
    max_samples=6
    
    # -----------------------------
    # Filter table
    # -----------------------------
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ].copy()

    # -----------------------------
    # Summarize counts
    # -----------------------------
    summary = (
        df.groupby(col, as_index=False)
          .agg(
              n_variants=("AA_sequence", "nunique"),
              mean_log2_enrichment=("Log2_enrichment", "mean")
          )
    )

    # Define x-range
    if include_zero:
        full_range = range(0, max_samples + 1)
    else:
        full_range = range(1, max_samples + 1)

    summary = (
        summary.set_index(col)
               .reindex(full_range)
               .fillna(0)
               .reset_index()
    )

    total_variants = summary["n_variants"].sum()

    summary["percent_variants"] = 100 * summary["n_variants"] / total_variants

    # -----------------------------
    # Styling
    # -----------------------------
    sns.set_style("whitegrid")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7, 4))

    bars = ax.bar(
        summary[col],
        summary["percent_variants"],
        color="#4C78A8",
        edgecolor="black",
        linewidth=1
    )

    # -----------------------------
    # Labels and title
    # -----------------------------
    ax.set_xlabel("Number of biological samples in which variant is detected")
    ax.set_ylabel("Percent of measured variants (%)")

    if title:
        ax.set_title(
            f"Variants detected across {extraction} {tissue} samples"
        )

    ax.set_xticks(summary[col])

    # Clean up spines and grid
    ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)

    # -----------------------------
    # Add absolute counts above bars
    # -----------------------------
    ymax = summary["percent_variants"].max()
    offset = ymax * 0.02 if ymax > 0 else 0.5

    for bar, n in zip(bars, summary["n_variants"]):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            f"{int(n):,}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    #  expand ylim so text fits and it doesnt look to crowded
    ax.set_ylim(0, ymax * 1.15 if ymax > 0 else 1)

    plt.tight_layout()

    # -----------------------------
    # Save
    # -----------------------------
    if save:
        
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            zero_str = "with0" if include_zero else "no0"
            file_name = f"variant_presence_distribution_{extraction}_{tissue}_{zero_str}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=600, bbox_inches="tight")

    plt.show()

    return summary

### 3.2. Boxplot for log2_enrichment distribution across variant presence

In [ ]:
def log2_enrichment_distribution_by_sample_presence(
    table,
    tissue,
    extraction,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
    # Definition of n_sample column
    col="n_samples_present"

    #filter table for tissue and extraction combination
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction) &
        (table[col].isin([1, 2, 3, 4, 5, 6]))
    ].copy()

    # set style of figure
    sns.set_style("whitegrid")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7, 4))

    # plot figure
    sns.boxplot(
        data=df,
        x=col,
        y="Log2_enrichment",
        order=[1, 2, 3, 4, 5, 6],
        color="#4C78A8",
        showfliers=False,
        ax=ax
    )

    # label of axis
    ax.set_xlabel("Number of biological samples in which variant is detected")
    ax.set_ylabel("Log2 enrichment")

    # title
    if title:
        ax.set_title(f"{extraction} {tissue}: log2 enrichment by sample presence")


    # background and axis settings
    ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"log2_enrichment_by_sample_presence_{extraction}_{tissue}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=400, bbox_inches="tight")

    plt.show()

### 3.3. Boxplot of counts distribution across variant presence

In [ ]:
def RPM_count_distribution_by_sample_presence(
    table,
    tissue,
    extraction,
    value = 'RPM',
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
    # Definition of n_sample column
    col="n_samples_present"

    #filter table for tissue and extraction combination
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction) &
        (table[col].isin([0, 1, 2, 3, 4, 5, 6]))
    ].copy()

    summary = (df.groupby([col], as_index = False)
        .agg({
            "Log2_enrichment": "mean",
            value: "mean"
            
        })
              )
    
    # set style of figure
    sns.set_style("whitegrid")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7, 4))

    # plot figure
    sns.boxplot(
        data=df,
        x=col,
        y=value,
        order=[0, 1, 2, 3, 4, 5, 6],
        color="#4C78A8",
        showfliers=False,
        ax=ax
    )

    # label of axis
    ax.set_xlabel("Number of biological samples in which variant is detected")
    ax.set_ylabel(f"Mean {value}")


    # title
    if title:
        ax.set_title(f"{extraction} {tissue}: Mean {value} by sample presence")


    # background and axis settings
    ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.grid(axis="x", visible=False)
    ax.set_yscale("log", base=10)
    sns.despine(ax=ax)
    
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"Mean_{value}_by_sample_presence_{extraction}_{tissue}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=400, bbox_inches="tight")

    plt.show()
    return summary

## 4. Load files and extract information

### 4.1. Load csv files

In [ ]:
%%time
#load long table
df_long = read_csv_file(tables_dir / 'summary', "df_long_combined_biological_technical_rep")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_sample_pooled")

In [ ]:
print ('Long Table')
display (df_long)

print ('Pooled Table')
display (df_pooled)

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

In [ ]:
display (SAMPLE, EXT, TISSUE, SEX, MOUSE_ID)

In [ ]:
# Sort different file names in extraction and biological or technical Replicates
DICT_NAMES = sort_file_names (SAMPLE)
DICT_NAMES

### 4.3. Load tissue/ext specific librarys¶

In [ ]:
%%time
# Load tissue specific library 
dict_library = {}
for tissue in TISSUE:
    df = read_csv_file(tables_dir/tissue, f'df_{tissue}_input_library')
    dict_library[tissue] = df

# Load raw library  for corr with special library
df_raw_input =  read_csv_file(tables_dir, 'df_input_lib_raw')

## 5. Figures

### 5.1. variant presence distribution across variant presence

In [ ]:
for tis, ext in product (TISSUE, EXT):
    print (ext, tis)
    summary = plot_variant_presence_distribution(
        table=df_pooled,
        tissue= tis,
        extraction= ext,
        include_zero=False,
        title=False,
        save=True,
        output_path=figures_dir/'1_proportion'/'5_variant_present_in_n_samples'
    )


### 5.2. Boxplot for log2_enrichment distribution across variant presence

### 5.3. Boxplot of counts distribution across variant presence

In [ ]:
for tis, ext in product (TISSUE, EXT):
    summary = RPM_count_distribution_by_sample_presence(table=df_pooled, tissue=tis, extraction=ext, value='RPM', save = True, output_path=figures_dir/'1_Mean_RPM_across_n_samples')
